# Ontologies to use
- **Post-Subreddit:** A post belongs to a specific subreddit.
- **Post-Author:** A post is created by an author.
- **Post-Topics:** A post is related to one or more topics based on the topic modeling results.
- **Post-Comments:** A post has a set of comments.
- **Post-Keywords:** A post is associated with specific keywords derived from the topic modeling.

## Loading libraries to be used

In [1]:
# Download NLTK resources 
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
!pip install pyvis



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import json
import rdflib
import gensim
import pandas as pd
import networkx as nx
from gensim import corpora
from pymongo import MongoClient
from nltk.corpus import stopwords
import plotly.graph_objects as go
from pyvis.network import Network
from collections import defaultdict
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

## Reading data from *MongoDB*

- Connect to MongoDB
- Read the data from the collection
- Convert the data into a pandas DataFrame
- Print the first few rows of the DataFrame

In [3]:
client = MongoClient("mongodb://localhost:27017/")
db = client['DataTails']
collection = db['Data']  
data_cursor = collection.find({})
DF = pd.DataFrame(list(data_cursor))
print(DF.head())

                        _id  type  \
0  66e965e698330736e0d693d5  None   
1  66e965e698330736e0d693d6  None   
2  66e965e698330736e0d693d7  None   
3  66e965e698330736e0d693d8  None   
4  66e965e698330736e0d693d9  None   

                                           postTitle postDesc  \
0  Adults(especially those over 30), how young do...      NaN   
1  What is a thing that your parents consider nor...      NaN   
2                 What is a smell that comforts you?      NaN   
3  When in history was it the best time to be a w...      NaN   
4    What's the worst way someone broke up with you?      NaN   

              postTime            authorName noOfUpvotes isNSFW  \
0  2024-08-06 01:02:35  Excellent-Studio7257        4068  False   
1  2024-08-06 01:47:22        Bigbumoffhappy        2073  False   
2  2024-08-05 22:21:53         bloomsmittenn        2188  False   
3  2024-08-06 03:32:59   More_food_please_77         778  False   
4  2024-08-05 21:01:39    ImpressiveWrap7363       

## Preprocessing the Data

- **Stopword Removal & Lemmatization:** 
    - Preprocessing() uses NLTK to remove stopwords and lemmatize the text.
- **Handling Missing Data:**
    - Missing postDesc fields are filled with an empty string.
    - Missing noOfUpvotes is filled with 0.
- **Datetime Conversion:** 
    - postTime is converted to a datetime object, and any errors are coerced.
- **Handling Comments:** 
    - Comments are converted from list format to a string of concatenated comments.
- **Final Clean Text:** 
    - Both postTitle and postDesc are cleaned using regular expressions and tokenization, and then passed through the NLTK-based text preprocessing function.

In [4]:
lemmatizer = WordNetLemmatizer()
StopWords = set(stopwords.words('english'))
custom_stopwords = StopWords | {"make", "thing", "know", "get", "want", "like", "would", "could", "you", "say","also","aita","com","www","made","ago","day","000"}
def Preprocessing(text):
    text = re.sub(r'\W', ' ', text)  # Remove non-alphanumeric characters
    text = re.sub(r'\b\w{1,2}\b', '', text)  # Remove short words
    tokens = word_tokenize(text.lower())
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in custom_stopwords and len(word) > 2]
    return tokens

Cols = ['subReddit', 'postTitle', 'postDesc', 'postTime', 'authorName', 'noOfUpvotes', 'comments', 'noOfComments', 'postUrl','imageUrl','isNSFW']
DF = DF[Cols]
print(DF.isnull().sum())


DF['subReddit'] = DF['subReddit'].fillna('Unknown_SubReddit')
DF['authorName'] = DF['authorName'].fillna('Unknown_Author')
DF['postTitle'] = DF['postTitle'].fillna('Untitled')
DF['postUrl'] = DF['postUrl'].fillna('http://example.com/NOPOST.png')
DF['imageUrl'] = DF['imageUrl'].fillna('http://example.com/NOImage.png')
DF['isNSFW'] = DF['isNSFW'].fillna(False)
DF['postDesc'].fillna('', inplace=True)
DF['noOfUpvotes'].fillna(0, inplace=True)
DF['noOfComments'].fillna(0, inplace=True)
DF['postTime'] = pd.to_datetime(DF['postTime'], errors='coerce')
DF['comments'] = DF['comments'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))
DF['postTitle'] = DF['postTitle'].apply(lambda x: Preprocessing(str(x)))
DF['postDesc'] = DF['postDesc'].apply(lambda x: Preprocessing(str(x)))
DF['isNSFW'] = DF['isNSFW'].astype(bool)
print(DF.head())
print(DF.isnull().sum())

subReddit           2
postTitle           0
postDesc        15607
postTime            0
authorName        587
noOfUpvotes         0
comments            0
noOfComments        2
postUrl             2
imageUrl            3
isNSFW              1
dtype: int64
   subReddit                                          postTitle postDesc  \
0  AskReddit                   [adult, especially, young, seem]       []   
1  AskReddit  [parent, consider, normal, consider, normal, a...       []   
2  AskReddit                                   [smell, comfort]       []   
3  AskReddit                       [history, best, time, woman]       []   
4  AskReddit                       [worst, way, someone, broke]       []   

             postTime            authorName noOfUpvotes  \
0 2024-08-06 01:02:35  Excellent-Studio7257        4068   
1 2024-08-06 01:47:22        Bigbumoffhappy        2073   
2 2024-08-05 22:21:53         bloomsmittenn        2188   
3 2024-08-06 03:32:59   More_food_please_77         

In [5]:
import rdflib
from rdflib import URIRef, Literal
from collections import defaultdict
import gensim
from gensim import corpora
import json

# RDF graph initialization
g = rdflib.Graph()
SIOC = rdflib.Namespace("http://rdfs.org/sioc/ns#")
DCMI = rdflib.Namespace("http://purl.org/dc/elements/1.1/")
FOAF = rdflib.Namespace("http://xmlns.com/foaf/0.1/")
g.bind("sioc", SIOC)
g.bind("dc", DCMI)
g.bind("foaf", FOAF)

# Function to add post data to RDF graph
def add_post_to_graph(row, index, topic_uri, subreddit_type_uri, author_type_uri, post_type_uri, comment_type_uri):
    post_uri = URIRef(f"http://reddit.com/post{index}")
    subreddit_uri = URIRef(f"http://reddit.com/subreddit/{row['subReddit']}")
    author_uri = URIRef(f"http://reddit.com/user/{row['authorName']}")
    
    # Add post properties and link to topic
    g.add((post_uri, rdflib.RDF.type, SIOC.Post))
    g.add((post_uri, DCMI.title, Literal(' '.join(row['postTitle']))))
    g.add((post_uri, DCMI.description, Literal(' '.join(row['postDesc']))))
    g.add((post_uri, DCMI.date, Literal(row['postTime'])))
    g.add((post_uri, SIOC.num_replies, Literal(row['noOfUpvotes'])))
    g.add((post_uri, SIOC.link, URIRef(row['postUrl'])))
    g.add((post_uri, SIOC.NSFW, Literal(row['isNSFW'])))
    g.add((post_uri, SIOC.has_type, post_type_uri))
    g.add((post_uri, SIOC.topic, topic_uri))

    # Link post to subreddit and type
    g.add((subreddit_uri, rdflib.RDF.type, SIOC.Container))
    g.add((subreddit_uri, SIOC.has_post, post_uri))
    g.add((subreddit_uri, SIOC.has_type, subreddit_type_uri))
    g.add((post_uri, SIOC.Container, subreddit_uri))

    # Link to author and author type
    g.add((author_uri, rdflib.RDF.type, FOAF.Person))
    g.add((author_uri, FOAF.name, Literal(row['authorName'])))
    g.add((post_uri, SIOC.has_creator, author_uri))
    g.add((author_uri, SIOC.has_type, author_type_uri))

    # Add comments and link to comment type
    comment_uri = URIRef(f"http://reddit.com/comment/{index}")
    g.add((comment_uri, rdflib.RDF.type, SIOC.Comment))
    g.add((comment_uri, DCMI.title, Literal(row['comments'])))
    g.add((post_uri, SIOC.has_reply, comment_uri))
    g.add((comment_uri, SIOC.has_type, comment_type_uri))

# Process each subreddit and create LDA models
GroupedData = DF.groupby('subReddit')
all_topics = defaultdict(set)

for subreddit, group in GroupedData:
    group['Combined'] = group['postTitle'] + group['postDesc']
    group['Tokens'] = group['Combined'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x if word not in custom_stopwords])

    # Create dictionary and corpus
    dictionary = corpora.Dictionary(group['Tokens'])
    corpus = [dictionary.doc2bow(text) for text in group['Tokens']]
    LDA = gensim.models.LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15, iterations=100, random_state=42)
    
    for idx, topic in LDA.print_topics(num_words=5):
        topic_uri = URIRef(f"http://reddit.com/topic/{subreddit}_{idx}")
        topic_words = topic.split(" + ")
        unique_words = {word.split("*")[1].strip('"') for word in topic_words}
        all_topics[subreddit].update(unique_words)

        # Define URIs for types
        subreddit_type_uri = URIRef(f"http://reddit.com/type/subreddit/{subreddit}")
        post_type_uri = URIRef(f"http://reddit.com/type/post/{subreddit}")
        comment_type_uri = URIRef(f"http://reddit.com/type/comment/{subreddit}")
        author_type_uri = URIRef(f"http://reddit.com/type/author/{subreddit}")

        # Link subreddit to topic
        subreddit_uri = URIRef(f"http://reddit.com/subreddit/{subreddit}")
        g.add((subreddit_uri, SIOC.has_topic, topic_uri))
        g.add((topic_uri, rdflib.RDF.type, SIOC.Topic))

        # Iterate through each row in the group
        for index, row in group.iterrows():
            add_post_to_graph(row, index, topic_uri, subreddit_type_uri, author_type_uri, post_type_uri, comment_type_uri)

# Save graph
g.serialize('D:/FYP/Github/data-tails/Backend/Ontologies/KG.ttl', format='turtle')
json_ld = g.serialize(format='json-ld', indent=4)
with open("D:/FYP/Github/data-tails/Backend/Ontologies/KG.json", "w", encoding="utf-8") as f:
    f.write(json_ld)

### Converting *KG.json, KG.ttl* to format of D3 input file to view graph

In [6]:
# Generate the D3-compatible JSON structure
d3_data = {
    "nodes": [],
    "links": []
}

# Create a mapping for nodes to avoid duplicates
node_map = {}

# Function to add a node if it doesn't already exist
def add_node(label, node_type, description=""):
    if label not in node_map:
        node_id = len(d3_data["nodes"])
        node_map[label] = node_id
        d3_data["nodes"].append({"id": node_id, "label": label, "type": node_type, "description": description})
    return node_map[label]

# Populate nodes and links based on the RDF graph structure
for subreddit in g.subjects(rdflib.RDF.type, SIOC.Container):
    subreddit_label = str(subreddit)
    subreddit_id = add_node(subreddit_label, "subreddit", description="Community for specific topics")

    # For each post in the subreddit
    for post in g.objects(subreddit, SIOC.has_post):
        post_label = str(post)
        post_id = add_node(post_label, "post", description="Individual posts in the subreddit")

        # Create a link from subreddit to post
        d3_data["links"].append({"source": subreddit_id, "target": post_id})

        # Add authors
        author = g.value(post, SIOC.has_creator)
        if author:
            author_label = str(author)
            author_id = add_node(author_label, "author", description="User who created the post")

            # Create a link from post to author
            d3_data["links"].append({"source": post_id, "target": author_id})

        # For each comment on the post
        for comment in g.objects(post, SIOC.has_reply):
            comment_label = str(comment)
            comment_id = add_node(comment_label, "comment", description="User comments on the post")

            # Create a link from post to comment
            d3_data["links"].append({"source": post_id, "target": comment_id})

            # Link comment to author if available
            if author:
                d3_data["links"].append({"source": author_id, "target": comment_id})

# Write the D3-compatible JSON data to a file
with open("D:/FYP/Github/data-tails/Backend/Ontologies/D3KG.json", "w", encoding="utf-8") as f:
    json.dump(d3_data, f, indent=4)
